# Suffix Trees

A **Suffix Tree** is one of the most powerful data structures for string processing. It's essentially a compressed trie of all suffixes of a given string, enabling extremely efficient pattern matching and many other string operations.

---

[← Previous: Suffix Arrays](../../Tier_4_Algorithms_and_Data_Structures/08_Advanced_String_Structures/03_suffix_arrays.ipynb)

---

## 1. What is a Suffix Tree?

A **Suffix Tree** for a string `S` of length `n` is a compressed trie containing all `n` suffixes of `S`.

### Key Properties

1. **Exactly `n` leaves** - one for each suffix (positions 0 to n-1)
2. **At most `n-1` internal nodes** (excluding root) - total O(n) nodes
3. **Each internal node (except root) has ≥2 children**
4. **Edge labels are substrings** (not single characters like in a trie)
5. **Path from root to each leaf spells out one suffix**
6. **No two edges from the same node start with the same character**

### Suffix Tree vs Suffix Trie

| Feature | Suffix Trie | Suffix Tree |
|---------|-------------|-------------|
| Edge labels | Single characters | Substrings |
| Number of nodes | O(n²) | O(n) |
| Space complexity | O(n²) | O(n) |
| Internal nodes | May have 1 child | Always ≥2 children |

## 2. ASCII Art Visualization

### Suffix Tree for "banana$"

First, let's list all suffixes:
```
Position 0: banana$
Position 1: anana$
Position 2: nana$
Position 3: ana$
Position 4: na$
Position 5: a$
Position 6: $
```

The suffix tree structure:
```
                        (root)
                    /    |     \
                   $    a       banana$
                  [6]   |          [0]
                        |
                       (a)
                      /   \
                    $     na
                   [5]     |
                          (na)
                         /    \
                        $     na$
                       [3]    [1]
                        
                        |
                    (also from root)
                        |
                       na
                        |
                       (na)
                      /    \
                     $     na$
                    [4]    [2]

Numbers in brackets [i] = starting position of that suffix
```

**Full tree structure:**
```
                            (root)
                    /     /    \      \
                   $    a      banana$  na
                  [6]   |        [0]     |
                       (a)              (na)
                      /   \            /    \
                    $     na          $     na$
                   [5]     |         [4]    [2]
                          (na)
                         /    \
                        $     na$
                       [3]    [1]
```

### Pattern Search Example: Finding "ana"

```
Step 1: Start at root, find edge starting with 'a'
        
            (root)
               |
               a    ← follow 'a'
               |
              (a)
             /   \
           $     na    ← follow 'n', then 'a' 
          [5]     |
                 (na)
                /    \
               $     na$
              [3]    [1]

Step 2: After following 'a' from root, we're at node (a)
Step 3: Find edge starting with 'n' → edge "na"
Step 4: Match 'n' and 'a' against edge "na" → complete match!

Pattern "ana" found as: root → 'a' → "na" (first 2 chars)

Step 5: Collect all leaves below current position:
        - Leaf [3] → suffix "ana$" starts at position 3
        - Leaf [1] → suffix "anana$" starts at position 1

Result: Pattern "ana" occurs at positions 1 and 3

Verification: "banana"
              0123456
              - Position 1: "anana" contains "ana" ✓
              - Position 3: "ana" matches exactly ✓
```

### Longest Common Substring (LCS) Visualization

```
S1 = "ABAB"
S2 = "BABA"

Combined string: "ABAB$BABA#"
($ terminates S1, # terminates S2)

Build suffix tree and mark leaves:
- Leaves with positions 0-4 belong to S1 (green)
- Leaves with positions 5-9 belong to S2 (red)
- Internal nodes with both green and red descendants are "blue"

Conceptual tree:

                    (root)
                   /  |  |  \
                  $  AB  BA  #
                 [4]  |   |  [9]
                     ...  ...

Find deepest internal node that has descendants from BOTH strings:
- Node with path "ABA" has leaves from both S1 and S2 → "blue"
- Node with path "BAB" has leaves from both S1 and S2 → "blue"

Deepest blue node depth = 3

LCS = "ABA" or "BAB" (both length 3)
```

## 3. Construction Methods

### Naive Construction - O(n²)

**Algorithm:**
1. Build a suffix trie by inserting all suffixes
2. Compress paths with single children into single edges

**Time Complexity:** O(n²) - inserting all suffixes takes quadratic time

### Ukkonen's Algorithm - O(n)

**Key Ideas:**
1. Build tree incrementally, left to right
2. Use suffix links to avoid redundant traversals
3. Implicit suffix tree becomes explicit with terminal character

**Optimizations:**
- **Suffix links:** Jump between nodes representing related suffixes
- **Edge compression:** Store (start, end) indices instead of substrings
- **Skip/count trick:** Jump over edges when length is known

### Complexity Comparison

| Operation | Naive | Ukkonen |
|-----------|-------|--------|
| Construction | O(n²) | O(n) |
| Space | O(n) | O(n) |
| Pattern search | O(m) | O(m) |
| All k occurrences | O(m + k) | O(m + k) |
| Longest repeated substring | O(n) | O(n) |
| LCS of two strings | O(n + m) | O(n + m) |

## 4. Implementation

### Node and Tree Classes

In [ ]:
class SuffixTreeNode:
    """
    A node in the suffix tree.
    
    Attributes:
        children: Dictionary mapping edge labels (strings) to child nodes
        suffix_index: For leaves, the starting position of the suffix (-1 for internal nodes)
        string_id: Identifier for which string this leaf belongs to (for LCS)
    """
    
    def __init__(self):
        self.children = {}      # edge_label -> SuffixTreeNode
        self.suffix_index = -1  # Starting position of suffix (-1 for internal nodes)
        self.string_id = None   # Which string this suffix belongs to (for LCS)
    
    def is_leaf(self):
        """Returns True if this is a leaf node (represents a complete suffix)."""
        return len(self.children) == 0
    
    def __repr__(self):
        if self.is_leaf():
            return f"Leaf[{self.suffix_index}]"
        return f"Internal(children={list(self.children.keys())})"

In [ ]:
class SuffixTree:
    """
    Suffix Tree implementation with naive O(n²) construction.
    
    The tree is built by first constructing a suffix trie (all suffixes
    inserted character by character), then compressing single-child paths
    into single edges with substring labels.
    
    Attributes:
        text: The original text (with terminal character)
        root: The root node of the suffix tree
    """
    
    def __init__(self, text, terminal='$'):
        """
        Build a suffix tree for the given text.
        
        Args:
            text: The input string
            terminal: Terminal character to append (default '$')
        """
        self.text = text + terminal
        self.root = SuffixTreeNode()
        self._build_trie()
        self._compress(self.root)
    
    def _build_trie(self):
        """
        Build a suffix trie by inserting all suffixes.
        This is O(n²) as we insert n suffixes of average length n/2.
        """
        n = len(self.text)
        for i in range(n):
            # Insert suffix starting at position i
            current = self.root
            for j in range(i, n):
                char = self.text[j]
                if char not in current.children:
                    current.children[char] = SuffixTreeNode()
                current = current.children[char]
            # Mark the leaf with its starting position
            current.suffix_index = i
    
    def _compress(self, node):
        """
        Compress single-child paths into single edges.
        This converts the trie into a proper suffix tree.
        """
        # Keep compressing until no more single-child internal nodes
        changed = True
        while changed:
            changed = False
            keys = list(node.children.keys())
            for key in keys:
                child = node.children[key]
                # If child has exactly one child, merge the edges
                if len(child.children) == 1:
                    grandchild_key = list(child.children.keys())[0]
                    grandchild = child.children[grandchild_key]
                    # Create new merged edge label
                    new_key = key + grandchild_key
                    node.children[new_key] = grandchild
                    del node.children[key]
                    changed = True
        
        # Recursively compress children
        for child in node.children.values():
            self._compress(child)
    
    def search(self, pattern):
        """
        Search for all occurrences of a pattern in the text.
        
        Args:
            pattern: The pattern to search for
            
        Returns:
            List of starting positions where pattern occurs
        """
        positions = []
        node = self.root
        remaining = pattern
        
        while remaining:
            # Find edge starting with first character of remaining pattern
            found = False
            for edge_label, child in node.children.items():
                if edge_label[0] == remaining[0]:
                    found = True
                    # Check how much of edge matches remaining pattern
                    match_len = min(len(edge_label), len(remaining))
                    if edge_label[:match_len] == remaining[:match_len]:
                        remaining = remaining[match_len:]
                        node = child
                    else:
                        # Mismatch within edge
                        return []
                    break
            
            if not found:
                return []
        
        # Collect all leaf positions below current node
        self._collect_leaves(node, positions)
        return sorted(positions)
    
    def _collect_leaves(self, node, positions):
        """Collect all suffix indices from leaves under this node."""
        if node.is_leaf():
            positions.append(node.suffix_index)
        else:
            for child in node.children.values():
                self._collect_leaves(child, positions)
    
    def visualize(self, node=None, prefix="", is_last=True, edge_label=""):
        """
        Print a visual representation of the suffix tree.
        """
        if node is None:
            node = self.root
            print("(root)")
        else:
            connector = "└── " if is_last else "├── "
            node_info = f"[{node.suffix_index}]" if node.is_leaf() else ""
            print(f"{prefix}{connector}\"{edge_label}\" {node_info}")
        
        children = list(node.children.items())
        for i, (edge, child) in enumerate(children):
            extension = "    " if is_last else "│   "
            self.visualize(child, prefix + extension, i == len(children) - 1, edge)
    
    def get_suffix_array(self):
        """
        Extract suffix array from the tree via lexicographic DFS traversal.
        
        Returns:
            List of suffix starting positions in lexicographic order
        """
        suffix_array = []
        
        def dfs(node):
            if node.is_leaf():
                suffix_array.append(node.suffix_index)
            else:
                # Visit children in lexicographic order of edge labels
                for edge_label in sorted(node.children.keys()):
                    dfs(node.children[edge_label])
        
        dfs(self.root)
        return suffix_array

### Basic Usage Examples

In [ ]:
# Build suffix tree for "banana"
text = "banana"
st = SuffixTree(text)

print(f"Suffix Tree for '{text}$':")
print()
st.visualize()

In [ ]:
# Pattern search examples
patterns = ["ana", "ban", "nan", "xyz", "a", "na"]

print(f"Pattern searches in '{text}':")
print("=" * 40)
for p in patterns:
    positions = st.search(p)
    if positions:
        print(f"  '{p}' found at positions: {positions}")
    else:
        print(f"  '{p}' not found")

In [ ]:
# Extract suffix array from tree
sa = st.get_suffix_array()
print(f"Suffix Array: {sa}")
print()
print("Suffixes in lexicographic order:")
for i, pos in enumerate(sa):
    print(f"  SA[{i}] = {pos}: '{text}$'[{pos}:] = '{text + '$'}[{pos}:]'")

## 5. Longest Repeated Substring

In [ ]:
class SuffixTreeLRS(SuffixTree):
    """
    Extended suffix tree with longest repeated substring functionality.
    
    The longest repeated substring corresponds to the deepest internal
    node in the suffix tree (a node with at least 2 children).
    """
    
    def find_longest_repeated_substring(self):
        """
        Find the longest repeated substring in the text.
        
        Returns:
            tuple: (substring, list of starting positions)
        """
        self.max_depth = 0
        self.lrs_node = None
        self.lrs_path = ""
        
        def dfs(node, depth, path):
            # Only consider internal nodes (with children)
            if not node.is_leaf() and len(node.children) >= 2:
                if depth > self.max_depth:
                    self.max_depth = depth
                    self.lrs_node = node
                    self.lrs_path = path
            
            for edge_label, child in node.children.items():
                dfs(child, depth + len(edge_label), path + edge_label)
        
        dfs(self.root, 0, "")
        
        if self.lrs_node is None:
            return ("", [])
        
        # Get all positions where LRS occurs
        positions = []
        self._collect_leaves(self.lrs_node, positions)
        
        return (self.lrs_path, sorted(positions))

In [ ]:
# Find longest repeated substring
text = "mississippi"
st = SuffixTreeLRS(text)

lrs, positions = st.find_longest_repeated_substring()
print(f"Text: '{text}'")
print(f"Longest Repeated Substring: '{lrs}'")
print(f"Occurs at positions: {positions}")
print()

# Verify
print("Verification:")
for pos in positions:
    print(f"  Position {pos}: '{text[pos:pos+len(lrs)]}'")

In [ ]:
# More examples
test_texts = ["banana", "abcabc", "aaaaaa", "abcdef"]

for text in test_texts:
    st = SuffixTreeLRS(text)
    lrs, positions = st.find_longest_repeated_substring()
    if lrs:
        print(f"'{text}': LRS = '{lrs}' at positions {positions}")
    else:
        print(f"'{text}': No repeated substring")

## 6. Longest Common Substring (LCS)

To find the LCS of two strings, we:
1. Concatenate them with different terminal characters: `S1$S2#`
2. Build a suffix tree
3. Mark leaves by which string they belong to
4. Find the deepest internal node with descendants from both strings

In [ ]:
class SuffixTreeLCS:
    """
    Suffix tree for finding Longest Common Substring between two strings.
    
    Uses the generalized suffix tree approach:
    - Concatenate strings with unique terminators: S1$S2#
    - Build suffix tree
    - Find deepest internal node with leaves from both strings
    """
    
    def __init__(self, text1, text2):
        """
        Build generalized suffix tree for two strings.
        
        Args:
            text1: First string
            text2: Second string
        """
        self.text1 = text1
        self.text2 = text2
        self.separator_pos = len(text1)  # Position of '$'
        self.combined = text1 + '$' + text2 + '#'
        self.root = SuffixTreeNode()
        
        self._build_trie()
        self._compress(self.root)
        self._mark_string_ids()
    
    def _build_trie(self):
        """Build suffix trie for combined string."""
        n = len(self.combined)
        for i in range(n):
            current = self.root
            for j in range(i, n):
                char = self.combined[j]
                if char not in current.children:
                    current.children[char] = SuffixTreeNode()
                current = current.children[char]
            current.suffix_index = i
    
    def _compress(self, node):
        """Compress single-child paths."""
        changed = True
        while changed:
            changed = False
            keys = list(node.children.keys())
            for key in keys:
                child = node.children[key]
                if len(child.children) == 1:
                    grandchild_key = list(child.children.keys())[0]
                    grandchild = child.children[grandchild_key]
                    new_key = key + grandchild_key
                    node.children[new_key] = grandchild
                    del node.children[key]
                    changed = True
        
        for child in node.children.values():
            self._compress(child)
    
    def _mark_string_ids(self):
        """
        Mark each leaf with which string it belongs to.
        
        - Suffixes starting at position <= separator_pos belong to string 1
        - Suffixes starting at position > separator_pos belong to string 2
        """
        def mark(node):
            if node.is_leaf():
                if node.suffix_index <= self.separator_pos:
                    node.string_id = 1  # Belongs to text1
                else:
                    node.string_id = 2  # Belongs to text2
            else:
                for child in node.children.values():
                    mark(child)
        
        mark(self.root)
    
    def _get_string_ids(self, node):
        """
        Get set of string IDs present in subtree rooted at node.
        """
        if node.is_leaf():
            return {node.string_id}
        
        ids = set()
        for child in node.children.values():
            ids |= self._get_string_ids(child)
        return ids
    
    def find_lcs(self):
        """
        Find the Longest Common Substring between the two strings.
        
        Returns:
            str: The longest common substring
        """
        self.max_depth = 0
        self.lcs = ""
        
        def dfs(node, depth, path):
            if node.is_leaf():
                return {node.string_id}
            
            # Collect string IDs from all children
            all_ids = set()
            for edge_label, child in node.children.items():
                child_ids = dfs(child, depth + len(edge_label), path + edge_label)
                all_ids |= child_ids
            
            # If this internal node has descendants from both strings
            if 1 in all_ids and 2 in all_ids:
                if depth > self.max_depth:
                    self.max_depth = depth
                    self.lcs = path
            
            return all_ids
        
        dfs(self.root, 0, "")
        return self.lcs

In [ ]:
# Find LCS of two strings
text1 = "ABAB"
text2 = "BABA"

st = SuffixTreeLCS(text1, text2)
lcs = st.find_lcs()

print(f"String 1: '{text1}'")
print(f"String 2: '{text2}'")
print(f"Longest Common Substring: '{lcs}'")
print(f"Length: {len(lcs)}")

In [ ]:
# More LCS examples
test_pairs = [
    ("ABABC", "BABCA"),
    ("ABCDEF", "XYZABC"),
    ("algorithm", "logarithm"),
    ("hello", "world"),
    ("programming", "grammar"),
]

print("LCS Examples:")
print("=" * 50)
for s1, s2 in test_pairs:
    st = SuffixTreeLCS(s1, s2)
    lcs = st.find_lcs()
    print(f"  LCS('{s1}', '{s2}') = '{lcs}' (length {len(lcs)})")

## 7. Comparison with Suffix Array

Both suffix trees and suffix arrays solve similar problems. Let's compare them.

In [ ]:
def build_suffix_array_naive(text):
    """
    Build suffix array using naive O(n² log n) approach.
    
    Returns list of starting positions of suffixes in lexicographic order.
    """
    text = text + '$'
    suffixes = [(text[i:], i) for i in range(len(text))]
    suffixes.sort()  # Sort by suffix string
    return [pos for suffix, pos in suffixes]


def search_suffix_array(text, pattern, sa):
    """
    Binary search for pattern in suffix array.
    Returns list of positions where pattern occurs.
    """
    text = text + '$'
    n = len(sa)
    m = len(pattern)
    
    # Find leftmost occurrence
    left = 0
    right = n - 1
    while left < right:
        mid = (left + right) // 2
        if text[sa[mid]:sa[mid]+m] < pattern:
            left = mid + 1
        else:
            right = mid
    first = left
    
    # Check if pattern exists
    if text[sa[first]:sa[first]+m] != pattern:
        return []
    
    # Find rightmost occurrence
    right = n - 1
    while left < right:
        mid = (left + right + 1) // 2
        if text[sa[mid]:sa[mid]+m] > pattern:
            right = mid - 1
        else:
            left = mid
    last = right
    
    return sorted(sa[first:last+1])

In [ ]:
# Compare suffix tree and suffix array
text = "mississippi"

# Build both structures
st = SuffixTree(text)
sa = build_suffix_array_naive(text)

print(f"Text: '{text}'")
print()
print("Suffix Array (from naive construction):")
print(f"  {sa}")
print()
print("Suffix Array (from suffix tree):")
print(f"  {st.get_suffix_array()}")
print()

# Compare search results
patterns = ["issi", "ss", "p", "miss"]
print("Pattern Search Comparison:")
print("-" * 50)
for p in patterns:
    st_result = st.search(p)
    sa_result = search_suffix_array(text, p, sa)
    match = "✓" if st_result == sa_result else "✗"
    print(f"  Pattern '{p}':")
    print(f"    Suffix Tree:  {st_result}")
    print(f"    Suffix Array: {sa_result} {match}")

### Performance Comparison

In [ ]:
import time
import random

def benchmark(sizes):
    """Benchmark suffix tree vs suffix array construction."""
    results = []
    
    for n in sizes:
        # Generate random DNA sequence
        text = ''.join(random.choice('ACGT') for _ in range(n))
        
        # Time suffix tree construction
        start = time.time()
        st = SuffixTree(text)
        st_time = time.time() - start
        
        # Time suffix array construction
        start = time.time()
        sa = build_suffix_array_naive(text)
        sa_time = time.time() - start
        
        results.append((n, st_time, sa_time))
        print(f"n={n:5d}: SuffixTree={st_time:.4f}s, SuffixArray={sa_time:.4f}s")
    
    return results

print("Construction Time Comparison (naive implementations):")
print("=" * 55)
benchmark([100, 500, 1000, 2000])

### When to Use What?

| Criterion | Suffix Tree | Suffix Array |
|-----------|-------------|---------------|
| **Construction** | O(n) with Ukkonen | O(n) with DC3/SA-IS |
| **Space** | ~20n bytes | ~5n bytes |
| **Pattern search** | O(m) | O(m log n) or O(m) with LCP |
| **Implementation** | Complex | Simpler |
| **Cache efficiency** | Poor (pointer chasing) | Good (contiguous) |
| **Practical speed** | Good for small alphabets | Often faster in practice |

**Choose Suffix Tree when:**
- Need O(m) pattern matching without LCP array
- Working with many dynamic queries
- Alphabet is small (DNA, small character set)
- Need to traverse suffix links

**Choose Suffix Array when:**
- Space is limited
- Implementation simplicity matters
- Building for large texts
- Cache efficiency is important

## 8. Applications Summary

### Pattern Matching
- Find all occurrences of pattern P in text T in O(|P| + k) time
- k = number of occurrences

### Longest Repeated Substring
- Find the deepest internal node
- O(n) time

### Longest Common Substring
- Build generalized suffix tree for both strings
- Find deepest node with leaves from both strings
- O(n + m) time

### Other Applications
1. **Longest palindromic substring**
2. **Approximate pattern matching**
3. **Tandem repeats detection**
4. **Text compression (LZ77/LZ78)**
5. **DNA sequence analysis**
6. **Plagiarism detection**

## 9. Complete Example: DNA Analysis

In [ ]:
# DNA sequence analysis example
dna_sequence = "ATCGATCGATCGAATTCCGATCG"

print("DNA Sequence Analysis")
print("=" * 50)
print(f"Sequence: {dna_sequence}")
print(f"Length: {len(dna_sequence)}")
print()

# Build suffix tree
st = SuffixTreeLRS(dna_sequence)

# Find repeated motifs
lrs, positions = st.find_longest_repeated_substring()
print(f"Longest Repeated Motif: '{lrs}'")
print(f"Occurs at positions: {positions}")
print()

# Search for specific motifs
motifs = ["ATCG", "AATTCC", "GATC", "CGA"]
print("Motif Search:")
for motif in motifs:
    positions = st.search(motif)
    print(f"  '{motif}': {positions if positions else 'not found'}")

In [ ]:
# Compare two DNA sequences
seq1 = "ATCGATCGAATTCC"
seq2 = "GAATTCCGATCGAT"

lcs_tree = SuffixTreeLCS(seq1, seq2)
lcs = lcs_tree.find_lcs()

print("DNA Sequence Comparison")
print("=" * 50)
print(f"Sequence 1: {seq1}")
print(f"Sequence 2: {seq2}")
print(f"Longest Common Region: '{lcs}' (length {len(lcs)})")
print(f"Similarity: {100 * len(lcs) / min(len(seq1), len(seq2)):.1f}%")

## 10. Summary

### Key Takeaways

1. **Suffix Tree = Compressed Trie of all suffixes**
   - O(n) space (vs O(n²) for suffix trie)
   - O(n) construction with Ukkonen's algorithm

2. **Powerful for string operations**
   - Pattern matching in O(m) time
   - Finding repeated substrings
   - Longest common substring

3. **Trade-offs**
   - More complex to implement than suffix arrays
   - Uses more memory in practice
   - Better query performance for some operations

4. **Real-world applications**
   - Bioinformatics (DNA/protein analysis)
   - Text editors (search & replace)
   - Data compression
   - Plagiarism detection

---

## Exercises

### Exercise 1: Longest Common Substring Between Two DNA Sequences (**)

Finding the longest common substring (LCS) between two sequences is central to pairwise alignment algorithms. The generalized suffix tree approach does it in O(n + m) time.

**Task:** Use `SuffixTreeLCS` to find the longest common substring between `SEQ1` and `SEQ2` below. Then write a brute-force verifier `lcs_brute(s1, s2)` that checks all pairs of substrings, and confirm both approaches return the same result.

Explore: what happens when you compare a sequence with its own reverse complement?

In [ ]:
SEQ1 = "ATCGATCGAATTCCGATCG"
SEQ2 = "GAATTCCGATCGATCGTAG"


def lcs_brute(s1: str, s2: str) -> str:
    """
    Find the longest common substring by checking all pairs of substrings.

    Args:
        s1, s2: Input strings

    Returns:
        The longest substring common to both strings (first one found if tied)
    """
    # YOUR CODE HERE
    pass


def reverse_complement(seq: str) -> str:
    """Return the reverse complement of a DNA sequence.""""
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]


# lcs_tree = SuffixTreeLCS(SEQ1, SEQ2)
# result_tree = lcs_tree.find_lcs()
# result_brute = lcs_brute(SEQ1, SEQ2)
# print(f"Suffix tree LCS: '{result_tree}' (length {len(result_tree)})")
# print(f"Brute force LCS: '{result_brute}' (length {len(result_brute)})")
# print(f"Results match: {len(result_tree) == len(result_brute)}")
#
# rc = reverse_complement(SEQ1)
# lcs_rc = SuffixTreeLCS(SEQ1, rc)
# print(f"\nLCS of SEQ1 with its reverse complement: '{lcs_rc.find_lcs()}'")

### Exercise 2: Finding All Maximal Repeats (**)

A **maximal repeat** is a repeated substring that cannot be extended in either direction while remaining a repeat. In genomics, maximal repeats correspond to conserved sequence elements and repetitive regions of interest.

**Task:** Implement `find_maximal_repeats(sequence)` using `SuffixTreeLRS` (or `SuffixTree`). A repeat is maximal if extending it left or right by one character makes it non-repeated. Return all maximal repeats sorted by length (longest first).

Approach: an internal node with depth `d` represents a repeat of length `d`. It is maximal if no parent internal node covers all the same leaf positions.

In [ ]:
REPEAT_SEQ = "ATCGATCGAATTCGATCGATCGTTAATCGATCG"


def find_maximal_repeats(sequence: str) -> list[tuple[str, list[int]]]:
    """
    Find all maximal repeats in a DNA sequence using the suffix tree.

    A substring is a maximal repeat if:
    - It occurs at least twice (internal node in the suffix tree)
    - It cannot be extended left or right while remaining a repeat

    Args:
        sequence: DNA string (terminal '$' will be appended internally)

    Returns:
        List of (repeat_string, [start_positions]) sorted by descending length
    """
    # YOUR CODE HERE
    # Hint: build SuffixTree(sequence), then traverse internal nodes.
    # Each internal node with >= 2 leaves is a candidate repeat.
    # The path label from root to that node is the repeated string.
    pass


# repeats = find_maximal_repeats(REPEAT_SEQ)
# print(f"Maximal repeats in '{REPEAT_SEQ}':")
# for repeat, positions in repeats:
#     print(f"  '{repeat}' (len {len(repeat)}) at positions {positions}")

---

[← Previous: Suffix Arrays](../../Tier_4_Algorithms_and_Data_Structures/08_Advanced_String_Structures/03_suffix_arrays.ipynb)